In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pyspark.sql.functions as F

orders = spark.read.table('databricks_prep.data.orders')
display(orders)
orders.printSchema()
orders.limit(10).display()   


In [0]:
df=orders.select("customer_id","order_id","category","unit_price")
display(df)

In [0]:
df1 = df.filter(df.category == "Electronics")
display(df1)

In [0]:
df2 = df.filter(df.unit_price >750)
display(df)

In [0]:
df1 = orders.filter(orders.status == "Completed")
display(df1)

## Total SALES

In [0]:
orders1 = orders.withColumn("Sales",col("quantity")*col("unit_price"))
display(orders1)

### Sales by Each Category

In [0]:

orders = orders.withColumn("Sales",col("quantity")*col("unit_price"))
orders.groupBy("category").agg(sum("sales").alias("total_sales")).show()
display(orders)


### Average salary by Catagory

In [0]:
orders.groupBy("category").agg(avg("sales").alias("avg_sales")).show()

### Max salary in each catagory

In [0]:
orders.groupBy("category").agg(max("sales").alias("max_sales")).show()

### Find number of orders per customer

In [0]:
orders.groupBy("customer_id").count().show()

In [0]:
customer_df = spark.read.table("databricks_prep.data.customers")
display(customer_df)

### Join orders with customers

In [0]:
info = orders.join(customer_df,orders.customer_id == customer_df.customer_id,"inner")
info1 = info.select("order_id","customer_name","city","category","unit_price")
display(info)

### Find total sales by customer

In [0]:
sale = info.withColumn("Sales", info.unit_price * info.quantity)
sale1 = sale.groupBy("customer_name").agg(sum("Sales").alias("total_sales"))
display(sale1)

### Find customers who never placed an order

In [0]:
info = customer_df.join(orders,customer_df.customer_id == orders.customer_id,"left_anti")
display(info)

# Window Functions

## Load employees

In [0]:
employees = spark.read.table("databricks_prep.data.employees")
display(employees)

### Rank employees based on salary

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

In [0]:
windowSpec = Window.orderBy(col("salary").desc())
ranked_employees = employees.withColumn("rank", rank().over(Window.orderBy(col("salary").desc())))
display(ranked_employees)

### Find highest-paid employee in each department

without Rank

In [0]:
# Find max salary per department
max_salary_per_dept = employees.groupBy("department").agg(max("salary").alias("max_salary"))

# Join back to get employee details
high = employees.join(max_salary_per_dept, 
                      (employees.department == max_salary_per_dept.department) & 
                      (employees.salary == max_salary_per_dept.max_salary), 
                      "inner").select(employees["*"])
display(high)

with rank()

In [0]:
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

In [0]:
max_salary_per_dept = employees.withColumn("row_number",row_number().over(window_spec)).filter(col("row_number") == 1)
display(max_salary_per_dept)

###  Find top 2 employees from each department

In [0]:
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
max_sal = employees.withColumn("row_number",row_number().over(window_spec)).filter(col("row_number")<=2)
display(max_sal)

In [0]:
%sql
select * from databricks_prep.data.employees

### Difference between employee salary and department average

In [0]:
avg_salary = employees.groupBy("department").agg(avg("salary").alias("avg_salary"))
avgd = employees.join(avg_salary,employees.department==avg_salary.department,"inner")
avg1 = avgd.withColumn("diff",col("salary")-col("avg_salary"))
display(avg1)



In [0]:
from pyspark.sql import functions as F
window_spec = Window.partitionBy("department")
df_resu = (employees.withColumn("dept_avg_salary",F.avg("salary").over(window_spec)).withColumn("salary_difference",F.col("salary")-F.col("dept_avg_salary")))
display(df_resu)
           

### Find second-highest salary in each department

In [0]:
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
high = employees.withColumn("dense_rank",dense_rank().over(window_spec)).filter(col("dense_rank")==3)
display(high)

# Data Cleaning

### Find null values in transaction table

In [0]:
transactions = spark.read.table("databricks_prep.data.transactions")
display(transactions)

In [0]:
null_value = transactions.filter(col("payment_method").isNull()|col("city").isNull() |col("amount").isNull() ).show()
print(null_value)

### Replace null payment method with "Unknown"

In [0]:
from pyspark.sql.functions import coalesce, lit

df = transactions.withColumn(
    "payment_method",
    coalesce(col("payment_method"), lit("Unknown"))
)
display(df)

In [0]:
transactions.dropDuplicates["transaction_id"].show()